In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the approved repository branch and record the commit
import os
import subprocess
from getpass import getpass
REPO_URL = 'https://github.com/YasinEkici/hyperkvasir-multi-cnn-fusion.git'
BRANCH = 'week3.5/perf-colab'
REPO_DIR = '/content/hyperkvasir-multi-cnn-fusion'
assert not os.path.exists(REPO_DIR), f'Refresh the runtime or remove stale checkout: {REPO_DIR}'
GITHUB_TOKEN = getpass('GitHub token for private repo clone (leave empty if repo is public): ')
clone_url = REPO_URL
if GITHUB_TOKEN:
    clone_url = REPO_URL.replace('https://', f'https://x-access-token:{GITHUB_TOKEN}@')
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', clone_url, REPO_DIR], check=True)
GITHUB_TOKEN = None
os.chdir(REPO_DIR)
!git rev-parse HEAD

In [ ]:
# 3. Build the isolated Colab environment and hard-assert an A100
!python -m pip install -q uv
!uv venv --python 3.11 .venv
!uv pip sync --python .venv/bin/python env/requirements-colab.txt
!uv run --no-sync python -c "import torch, torchvision; print('torch', torch.__version__); print('torchvision', torchvision.__version__); print('torch_cuda', torch.version.cuda); print('cuda_available', torch.cuda.is_available()); assert torch.cuda.is_available(), 'CUDA unavailable'; name=torch.cuda.get_device_name(0); print('device', name); assert 'A100' in name, f'A100 required, found {name}'; print(torch.ones(1, device='cuda'))"

In [ ]:
# 4. Set the Step 3 focal 5-fold run and write its immutable resolved configuration
import os
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive/hyperkvasir-multi-cnn-fusion'
os.environ['RUN_ID'] = 'colab_a100_exp16_focal_5fold'
os.environ['EXPERIMENT'] = '16_triple_weighted_finetune_focal_official'
os.environ['FOLDS'] = '0 1 2 3 4'
os.environ['TRAINING_CONFIG'] = 'configs/training/finetune_wide_focal.yaml'
!uv run --no-sync python scripts/write_resolved_config.py --experiment $EXPERIMENT --training $TRAINING_CONFIG --run-id $RUN_ID --device cuda --folds $FOLDS

In [ ]:
# 5. Stage the approved Drive dataset onto local Colab storage
!test -d "$DRIVE_ROOT/data/hyperkvasir/labeled-images"
!mkdir -p data/raw/hyperkvasir
!cp -a "$DRIVE_ROOT/data/hyperkvasir/labeled-images" data/raw/hyperkvasir/

In [ ]:
# 6. Run the hard-stop provenance gate before training
import os, subprocess
os.environ['EXPECTED_GIT_SHA'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
!uv run --no-sync python scripts/check_provenance.py --run-id $RUN_ID --source-dataset-root "$DRIVE_ROOT/data/hyperkvasir/labeled-images" --staged-dataset-root data/raw/hyperkvasir/labeled-images --manifest data/splits/hyperkvasir_official_5fold/fold_0.csv --approved-source "$DRIVE_ROOT/data/hyperkvasir/labeled-images" --expected-git-sha $EXPECTED_GIT_SHA --device cuda

In [ ]:
# 7. Run repository training only. The user executes this GPU cell.
!uv run --no-sync python scripts/run_cv.py --experiment $EXPERIMENT --folds $FOLDS --training $TRAINING_CONFIG --device cuda

In [ ]:
# 8. Collect the fixed artifact set back to Drive without overwrite
!uv run --no-sync python scripts/collect_outputs.py --run-id $RUN_ID --experiment $EXPERIMENT --folds $FOLDS --destination-root "$DRIVE_ROOT/returned_outputs"

In [ ]:
# 9. Final returned-artifact checklist
!test -f "$DRIVE_ROOT/returned_outputs/$RUN_ID/resolved_config.yaml"
!test -f "$DRIVE_ROOT/returned_outputs/$RUN_ID/runtime.json"
!test -f "$DRIVE_ROOT/returned_outputs/$RUN_ID/${RUN_ID}_provenance.json"
!for f in 0 1 2 3 4; do test -f "$DRIVE_ROOT/returned_outputs/$RUN_ID/fold_${f}/best.pt"; done
!for f in 0 1 2 3 4; do test -f "$DRIVE_ROOT/returned_outputs/$RUN_ID/fold_${f}/metrics.json"; done
!for f in 0 1 2 3 4; do test -f "$DRIVE_ROOT/returned_outputs/$RUN_ID/fold_${f}/predictions.npz"; done
!echo '[OK] returned artifact checklist complete'